In [2]:
import pandas as pd
from datetime import datetime
import warnings
import locale

def extract_excel_data(file_path, sheet_name='Hasil'):
    """
    Fungsi untuk mengekstrak data dari file Excel
    
    Parameters:
        file_path (str): Path/lokasi file Excel
        sheet_name (str): Nama sheet yang akan diekstrak (default: 'Hasil')
    
    Returns:
        DataFrame: Data yang telah diekstrak dalam bentuk pandas DataFrame
    """
    try:
        # Set locale untuk format angka
        locale.setlocale(locale.LC_NUMERIC, '')
        
        # Membaca file Excel
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            df = pd.read_excel(file_path, sheet_name=sheet_name)
        
        # Membersihkan nama kolom (jika ada spasi atau karakter khusus)
        df.columns = df.columns.str.strip()
        
        # Fungsi untuk konversi string ke float dengan handle koma desimal
        def convert_to_float(value):
            if isinstance(value, str):
                value = value.replace('Rp.', '').replace('.', '').replace(',', '.').strip()
                try:
                    return float(value)
                except ValueError:
                    return None
            return value
        
        # Konversi kolom harga ke numeric
        harga_columns = ['Harga', 'HargaAtas', 'OmsetMinimum']
        for col in harga_columns:
            if col in df.columns:
                df[col] = df[col].apply(convert_to_float)
        
        # Fungsi untuk konversi persentase
        def convert_percentage(value):
            if isinstance(value, str):
                value = value.replace('%', '').replace(',', '.').strip()
                try:
                    return float(value) / 100.0
                except ValueError:
                    return None
            return value
        
        # Konversi kolom persentase ke numeric (0-1)
        pct_columns = ['ValidOrder', 'PersentaseKenaikanPenjualan', 'PersentaseKenaikanReview']
        for col in pct_columns:
            if col in df.columns:
                df[col] = df[col].apply(convert_percentage)
        
        # Tambahkan timestamp ekstraksi
        df['ExtractionTimestamp'] = datetime.now()
        
        return df
    
    except Exception as e:
        print(f"Error saat mengekstrak data: {str(e)}")
        return None
    finally:
        # Reset locale ke default
        locale.setlocale(locale.LC_NUMERIC, 'C')

# [Fungsi analyze_data dan save_to_csv tetap sama seperti sebelumnya]

if __name__ == "__main__":
    # Ganti dengan path file Excel Anda
    excel_file = "POST-1745802753-0de128507e.xlsx"
    
    # Ekstrak data
    extracted_data = extract_excel_data(excel_file)
    
    if extracted_data is not None:
        # Analisis data
        analyze_data(extracted_data)
        
        # Simpan ke CSV (opsional)
        save_to_csv(extracted_data, "extracted_sales_data.csv")
        
        # Informasi tambahan
        print("\nProses ekstraksi selesai!")
        print(f"Total produk unik: {extracted_data['Kategori3'].nunique()}")
        print(f"Total kota pengiriman unik: {extracted_data['KotaPengiriman'].nunique()}")


Informasi Data:
Jumlah Baris: 9883
Jumlah Kolom: 18

5 Baris Pertama Data:
               Kategori1  Kategori2              Kategori3   Harga  HargaAtas  \
0  Handphone & Aksesoris  Aksesoris  USB & Lampu Handphone  1700.0     1700.0   
1  Handphone & Aksesoris  Aksesoris  USB & Lampu Handphone  1700.0     1700.0   
2  Handphone & Aksesoris  Aksesoris  USB & Lampu Handphone  1650.0     1650.0   
3  Handphone & Aksesoris  Aksesoris  USB & Lampu Handphone  1649.0     1649.0   
4  Handphone & Aksesoris  Aksesoris  USB & Lampu Handphone  1599.0     1599.0   

   OmsetMinimum  JumlahTerjual  JumlahReview  JumlahRating  ValidOrder  \
0      49811700          29301          7191      4.744888      0.2454   
1       9948400           5852           331      4.833837      0.0566   
2       2311650           1401           208      4.846154      0.1485   
3      20884585          12665           562      4.784314      0.0444   
4       5163171           3229           255      4.800000      0.0

In [3]:
import csv
import sqlite3
import os
from datetime import datetime

# Nama file CSV dan database SQLite
csv_file = 'extracted_sales_data.csv'  # File CSV hasil ekstraksi
db_file = 'sales_analysis.db'          # Database SQLite output

# Membuat koneksi ke database SQLite
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# Membuat tabel utama untuk data penjualan
cursor.execute('''
CREATE TABLE IF NOT EXISTS sales (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    kategori1 TEXT,
    kategori2 TEXT,
    kategori3 TEXT,
    harga REAL,
    harga_atas REAL,
    omset_minimum REAL,
    jumlah_terjual INTEGER,
    jumlah_review INTEGER,
    jumlah_rating REAL,
    valid_order REAL,
    kota_pengiriman TEXT,
    penjualan_sebelumnya INTEGER,
    kenaikan_penjualan INTEGER,
    persentase_kenaikan_penjualan REAL,
    review_sebelumnya INTEGER,
    kenaikan_review INTEGER,
    persentase_kenaikan_review REAL,
    extraction_timestamp TEXT
)
''')

# Membuat tabel dimensi untuk kategori produk
cursor.execute('''
CREATE TABLE IF NOT EXISTS product_categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    kategori1 TEXT,
    kategori2 TEXT,
    kategori3 TEXT,
    UNIQUE(kategori1, kategori2, kategori3)
)
''')

# Membuat tabel dimensi untuk lokasi
cursor.execute('''
CREATE TABLE IF NOT EXISTS locations (
    location_id INTEGER PRIMARY KEY AUTOINCREMENT,
    kota_pengiriman TEXT UNIQUE
)
''')

# Fungsi untuk menambahkan data kategori produk
def insert_category(kategori1, kategori2, kategori3):
    try:
        cursor.execute('''
        INSERT OR IGNORE INTO product_categories (kategori1, kategori2, kategori3)
        VALUES (?, ?, ?)
        ''', (kategori1, kategori2, kategori3))
        conn.commit()
    except Exception as e:
        print(f"Error inserting category: {e}")

# Fungsi untuk menambahkan data lokasi
def insert_location(kota_pengiriman):
    try:
        cursor.execute('''
        INSERT OR IGNORE INTO locations (kota_pengiriman)
        VALUES (?)
        ''', (kota_pengiriman,))
        conn.commit()
    except Exception as e:
        print(f"Error inserting location: {e}")

# Fungsi untuk menambahkan data penjualan
def insert_sales(row):
    try:
        cursor.execute('''
        INSERT INTO sales (
            kategori1, kategori2, kategori3, harga, harga_atas, omset_minimum,
            jumlah_terjual, jumlah_review, jumlah_rating, valid_order,
            kota_pengiriman, penjualan_sebelumnya, kenaikan_penjualan,
            persentase_kenaikan_penjualan, review_sebelumnya, kenaikan_review,
            persentase_kenaikan_review, extraction_timestamp
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            row.get('Kategori1'),
            row.get('Kategori2'),
            row.get('Kategori3'),
            row.get('Harga'),
            row.get('HargaAtas'),
            row.get('OmsetMinimum'),
            row.get('JumlahTerjual'),
            row.get('JumlahReview'),
            row.get('JumlahRating'),
            row.get('ValidOrder'),
            row.get('KotaPengiriman'),
            row.get('PenjualanSebelumnya'),
            row.get('KenaikanPenjualan'),
            row.get('PersentaseKenaikanPenjualan'),
            row.get('ReviewSebelumnya'),
            row.get('KenaikanReviewPenjualan'),
            row.get('PersentaseKenaikanReview'),
            row.get('ExtractionTimestamp')
        ))
        conn.commit()
    except Exception as e:
        print(f"Error inserting sales data: {e}")
        print(f"Problematic row: {row}")

# Membaca data dari file CSV
try:
    with open(csv_file, mode='r', newline='', encoding='utf-8') as file:
        csv_reader = csv.DictReader(file)
        
        total_rows = 0
        success_rows = 0
        
        for row in csv_reader:
            total_rows += 1
            
            # Memasukkan data kategori produk
            insert_category(
                row.get('Kategori1'),
                row.get('Kategori2'),
                row.get('Kategori3')
            )
            
            # Memasukkan data lokasi
            insert_location(row.get('KotaPengiriman'))
            
            # Memasukkan data penjualan
            insert_sales(row)
            success_rows += 1
            
            # Progress report setiap 100 baris
            if total_rows % 100 == 0:
                print(f"Processed {total_rows} rows...")
        
        print(f"\nETL process completed. Success: {success_rows}/{total_rows} rows")
        
except FileNotFoundError:
    print(f"Error: File {csv_file} not found.")
except Exception as e:
    print(f"Error reading file {csv_file}: {e}")
finally:
    # Membuat indeks untuk mempercepat query
    print("\nCreating indexes for better performance...")
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_category ON sales(kategori1, kategori2, kategori3)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_location ON sales(kota_pengiriman)')
    conn.commit()
    
    # Menutup koneksi ke database
    conn.close()
    print("Database connection closed.")

Processed 100 rows...
Processed 200 rows...
Processed 300 rows...
Processed 400 rows...
Processed 500 rows...
Processed 600 rows...
Processed 700 rows...
Processed 800 rows...
Processed 900 rows...
Processed 1000 rows...
Processed 1100 rows...
Processed 1200 rows...
Processed 1300 rows...
Processed 1400 rows...
Processed 1500 rows...
Processed 1600 rows...
Processed 1700 rows...
Processed 1800 rows...
Processed 1900 rows...
Processed 2000 rows...
Processed 2100 rows...
Processed 2200 rows...
Processed 2300 rows...
Processed 2400 rows...
Processed 2500 rows...
Processed 2600 rows...
Processed 2700 rows...
Processed 2800 rows...
Processed 2900 rows...
Processed 3000 rows...
Processed 3100 rows...
Processed 3200 rows...
Processed 3300 rows...
Processed 3400 rows...
Processed 3500 rows...
Processed 3600 rows...
Processed 3700 rows...
Processed 3800 rows...
Processed 3900 rows...
Processed 4000 rows...
Processed 4100 rows...
Processed 4200 rows...
Processed 4300 rows...
Processed 4400 rows.

In [4]:
import sqlite3
from tabulate import tabulate

# Nama file database SQLite
db_file = 'sales_analysis.db'  # Sesuaikan dengan nama database hasil ETL

def display_table_data(table_name, limit=10):
    """Menampilkan data dari tabel tertentu dengan format yang rapi"""
    try:
        # Membuat koneksi ke database SQLite
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        
        # Mendapatkan nama kolom
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = [column[1] for column in cursor.fetchall()]
        
        # Menjalankan query untuk mengambil data
        cursor.execute(f'SELECT * FROM {table_name} LIMIT {limit}')
        rows = cursor.fetchall()
        
        # Menampilkan hasil dengan tabulate
        print(f"\nData dari tabel '{table_name}' (10 baris pertama):")
        print(tabulate(rows, headers=columns, tablefmt='grid'))
        
        # Menampilkan jumlah total baris
        cursor.execute(f'SELECT COUNT(*) FROM {table_name}')
        total_rows = cursor.fetchone()[0]
        print(f"\nTotal baris dalam tabel: {total_rows}")
        
    except sqlite3.Error as e:
        print(f"Error accessing database: {e}")
    finally:
        # Menutup koneksi ke database
        if conn:
            conn.close()

def show_database_summary():
    """Menampilkan ringkasan database"""
    try:
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        
        # Mendapatkan daftar tabel
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
        tables = cursor.fetchall()
        
        print("\nDaftar Tabel dalam Database:")
        print("=" * 30)
        for table in tables:
            table_name = table[0]
            cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
            count = cursor.fetchone()[0]
            print(f"- {table_name}: {count} records")
        
    except sqlite3.Error as e:
        print(f"Error accessing database: {e}")
    finally:
        if conn:
            conn.close()

if __name__ == "__main__":
    # Menampilkan ringkasan database
    show_database_summary()
    
    # Menampilkan data dari tabel-tabel utama
    display_table_data('sales')
    display_table_data('product_categories')
    display_table_data('locations')
    
    # Contoh query tambahan
    try:
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        
        print("\n10 Produk Terlaris:")
        cursor.execute('''
        SELECT kategori3, SUM(jumlah_terjual) as total_terjual 
        FROM sales 
        GROUP BY kategori3 
        ORDER BY total_terjual DESC 
        LIMIT 10
        ''')
        top_products = cursor.fetchall()
        print(tabulate(top_products, headers=['Produk', 'Total Terjual'], tablefmt='grid'))
        
        print("\n10 Kota dengan Penjualan Tertinggi:")
        cursor.execute('''
        SELECT kota_pengiriman, SUM(jumlah_terjual) as total_terjual 
        FROM sales 
        GROUP BY kota_pengiriman 
        ORDER BY total_terjual DESC 
        LIMIT 10
        ''')
        top_locations = cursor.fetchall()
        print(tabulate(top_locations, headers=['Kota', 'Total Terjual'], tablefmt='grid'))
        
    except sqlite3.Error as e:
        print(f"Error executing query: {e}")
    finally:
        if conn:
            conn.close()


Daftar Tabel dalam Database:
- sales: 9883 records
- sqlite_sequence: 3 records
- product_categories: 635 records
- locations: 178 records

Data dari tabel 'sales' (10 baris pertama):
+------+-----------------------+-------------+-----------------------+---------+--------------+-----------------+------------------+-----------------+-----------------+---------------+--------------------+------------------------+----------------------+---------------------------------+---------------------+-------------------+------------------------------+----------------------------+
|   id | kategori1             | kategori2   | kategori3             |   harga |   harga_atas |   omset_minimum |   jumlah_terjual |   jumlah_review |   jumlah_rating |   valid_order | kota_pengiriman    | penjualan_sebelumnya   | kenaikan_penjualan   | persentase_kenaikan_penjualan   | review_sebelumnya   | kenaikan_review   |   persentase_kenaikan_review | extraction_timestamp       |
+======+=======================+===